# Notebook 3 — Synthetic Fraud Generation & Augmentation

**Notebook 3 of 4.** For each imbalance level (1:100, 1:500, 1:1000), train each generator on the *training-set fraud only* and generate synthetic fraud up to a **balanced (1:1)** training set.

Generators: **CTGAN**, **TVAE** (deep), **SMOTE**, **ADASYN**, **Random Oversampling** (classical).

Outputs: one augmented training set per (ratio × generator), plus synthetic-quality metrics. Notebook 4 trains classifiers on these and evaluates on the held-out **real** test set.

> Reminder: synthetic data stays in TRAINING only. `val.parquet` / `test.parquet` are untouched real data.

In [ ]:
# Installs (SDV for CTGAN/TVAE, imbalanced-learn for SMOTE/ADASYN)
!pip install -q sdv imbalanced-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROC_DIR = '/content/drive/MyDrive/SDG_paper/data/processed'
AUG_DIR  = '/content/drive/MyDrive/SDG_paper/data/augmented'

import os
os.makedirs(AUG_DIR, exist_ok=True)

Mounted at /content/drive


In [ ]:
import pandas as pd, numpy as np, json, time, warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

EPOCHS = 100          # CTGAN / TVAE training epochs
RATIOS = [100, 500, 1000]

with open(os.path.join(PROC_DIR, 'feature_cols.json')) as f:
    FEATURES = json.load(f)
print(f'{len(FEATURES)} features')

76 features


## 1. Generator functions

Each takes the training set (real), learns the fraud class, and returns an **augmented** training set balanced to 1:1 (synthetic fraud added to training only). A plausibility filter removes impossible synthetic rows.

In [ ]:
from sdv.single_table import CTGANSynthesizer, TVAESynthesizer
from sdv.metadata import SingleTableMetadata
from imblearn.over_sampling import SMOTE, ADASYN, RandomOverSampler

def plausibility_filter(synth_df, real_fraud_df):
    """Drop synthetic rows outside the observed real-fraud range (per feature).
    Keeps generation honest; reports how many were removed."""
    lo = real_fraud_df[FEATURES].min()
    hi = real_fraud_df[FEATURES].max()
    # allow a small margin (10% of range) beyond observed min/max
    margin = (hi - lo).abs() * 0.10
    mask = ((synth_df[FEATURES] >= (lo - margin)) & (synth_df[FEATURES] <= (hi + margin))).all(axis=1)
    return synth_df[mask]

def augment_deep(train_df, synth_cls, n_needed, real_fraud):
    """CTGAN or TVAE: fit on fraud rows, sample, filter, top up if needed."""
    md = SingleTableMetadata(); md.detect_from_dataframe(real_fraud[FEATURES])
    model = synth_cls(metadata=md, epochs=EPOCHS, verbose=False)
    model.fit(real_fraud[FEATURES])
    collected, tries = [], 0
    while sum(len(c) for c in collected) < n_needed and tries < 6:
        s = model.sample(num_rows=int(n_needed * 1.5))
        s = plausibility_filter(s, real_fraud)
        collected.append(s); tries += 1
    synth = pd.concat(collected, ignore_index=True).head(n_needed)
    synth['label'] = 1
    return pd.concat([train_df, synth], ignore_index=True), synth

def augment_classical(train_df, sampler):
    """SMOTE / ADASYN / ROS: resample to balanced directly."""
    X, y = train_df[FEATURES], train_df['label']
    Xr, yr = sampler.fit_resample(X, y)
    out = pd.DataFrame(Xr, columns=FEATURES); out['label'] = yr
    return out

## 2. Synthetic-quality metrics

For the deep generators we record how well synthetic fraud matches real fraud: a Kolmogorov–Smirnov statistic per feature (lower = more similar) and a diversity measure (unique-row fraction). These feed the paper's quality discussion and flag the small-sample problem at 1:1000.

In [ ]:
from scipy.stats import ks_2samp

def quality_metrics(synth, real_fraud):
    ks = [ks_2samp(real_fraud[c], synth[c]).statistic for c in FEATURES]
    uniq = synth[FEATURES].round(4).drop_duplicates().shape[0] / max(1, len(synth))
    return {'ks_mean': float(np.mean(ks)), 'ks_max': float(np.max(ks)),
            'diversity_unique_frac': float(uniq), 'n_synth': int(len(synth))}

## 3. Run all (ratio × generator)

This is the heavy cell. For each ratio it loads the down-sampled training set, computes how much synthetic fraud is needed for balance, and runs each generator. Classical methods are fast; CTGAN/TVAE take the most time (≈ the bulk of runtime).

In [ ]:
quality_log = []

for r in RATIOS:
    tag = f'1_{r}'
    train = pd.read_parquet(os.path.join(PROC_DIR, f'train_{tag}.parquet'))
    real_fraud = train[train['label'] == 1].copy()
    n_legit = int((train['label'] == 0).sum())
    n_fraud = int((train['label'] == 1).sum())
    n_needed = n_legit - n_fraud   # to reach 1:1
    print(f'\n=== ratio 1:{r}  (legit={n_legit}, fraud={n_fraud}, need +{n_needed} synthetic) ===')

    # --- CTGAN ---
    t = time.time()
    aug, synth = augment_deep(train, CTGANSynthesizer, n_needed, real_fraud)
    aug.to_parquet(os.path.join(AUG_DIR, f'aug_{tag}_ctgan.parquet'))
    q = quality_metrics(synth, real_fraud); q.update({'ratio': r, 'gen': 'ctgan'}); quality_log.append(q)
    print(f'  CTGAN: +{len(synth)} synth, KS_mean={q["ks_mean"]:.3f}, div={q["diversity_unique_frac"]:.3f} ({time.time()-t:.0f}s)')

    # --- TVAE ---
    t = time.time()
    aug, synth = augment_deep(train, TVAESynthesizer, n_needed, real_fraud)
    aug.to_parquet(os.path.join(AUG_DIR, f'aug_{tag}_tvae.parquet'))
    q = quality_metrics(synth, real_fraud); q.update({'ratio': r, 'gen': 'tvae'}); quality_log.append(q)
    print(f'  TVAE:  +{len(synth)} synth, KS_mean={q["ks_mean"]:.3f}, div={q["diversity_unique_frac"]:.3f} ({time.time()-t:.0f}s)')

    # --- SMOTE / ADASYN / ROS ---
    for name, sampler in [('smote', SMOTE(random_state=RANDOM_STATE)),
                          ('adasyn', ADASYN(random_state=RANDOM_STATE)),
                          ('ros', RandomOverSampler(random_state=RANDOM_STATE))]:
        try:
            aug = augment_classical(train, sampler)
            aug.to_parquet(os.path.join(AUG_DIR, f'aug_{tag}_{name}.parquet'))
            print(f'  {name.upper()}: -> {len(aug)} rows')
        except Exception as e:
            print(f'  {name.upper()} FAILED: {e}')

    # --- baseline: no augmentation (just the imbalanced training set) ---
    train.to_parquet(os.path.join(AUG_DIR, f'aug_{tag}_none.parquet'))

# Save quality metrics for the paper
pd.DataFrame(quality_log).to_csv(os.path.join(AUG_DIR, 'synthetic_quality.csv'), index=False)
print('\nAll done. Quality metrics saved to synthetic_quality.csv')


=== ratio 1:100  (legit=656627, fraud=6566, need +650061 synthetic) ===
  CTGAN: +650061 synth, KS_mean=0.246, div=1.000 (839s)
  TVAE:  +650061 synth, KS_mean=0.206, div=1.000 (401s)
  SMOTE: -> 1313254 rows
  ADASYN: -> 1313394 rows
  ROS: -> 1313254 rows

=== ratio 1:500  (legit=656627, fraud=1313, need +655314 synthetic) ===
  CTGAN: +655314 synth, KS_mean=0.206, div=1.000 (330s)
  TVAE:  +655314 synth, KS_mean=0.223, div=1.000 (210s)
  SMOTE: -> 1313254 rows
  ADASYN: -> 1313186 rows
  ROS: -> 1313254 rows

=== ratio 1:1000  (legit=656627, fraud=657, need +655970 synthetic) ===
  CTGAN: +655970 synth, KS_mean=0.340, div=1.000 (251s)
  TVAE:  +655970 synth, KS_mean=0.242, div=1.000 (199s)
  SMOTE: -> 1313254 rows
  ADASYN: -> 1313310 rows
  ROS: -> 1313254 rows

All done. Quality metrics saved to synthetic_quality.csv


In [ ]:
# Review synthetic-quality table (watch the 1:1000 row — small-sample effects expected)
pd.DataFrame(quality_log)[['ratio', 'gen', 'n_synth', 'ks_mean', 'ks_max', 'diversity_unique_frac']]

,ratio,gen,n_synth,ks_mean,ks_max,diversity_unique_frac
0,100,ctgan,650061,0.245816,0.743321,1.0
1,100,tvae,650061,0.205835,0.597819,1.0
2,500,ctgan,655314,0.205784,0.537979,1.0
3,500,tvae,655314,0.223413,0.598779,1.0
4,1000,ctgan,655970,0.339633,0.864636,1.0
5,1000,tvae,655970,0.241542,0.515242,1.0


In [ ]:

from sklearn.neighbors import NearestNeighbors
import numpy as np, pandas as pd, os, json

with open(os.path.join(PROC_DIR, 'feature_cols.json')) as f:
    FEATURES = json.load(f)

def diversity_metrics(synth_fraud, real_fraud, sample=3000, seed=42):
    rng = np.random.RandomState(seed)
    rf = real_fraud[FEATURES].to_numpy()
    sf = synth_fraud[FEATURES].to_numpy()
    # subsample for speed
    if len(sf) > sample:
        sf = sf[rng.choice(len(sf), sample, replace=False)]
    nn = NearestNeighbors(n_neighbors=1).fit(rf)
    dist, idx = nn.kneighbors(sf)
    nn_dist_mean = float(dist.mean())
    # normalize by typical real-real spacing for comparability
    rr = NearestNeighbors(n_neighbors=2).fit(rf)
    rr_dist, _ = rr.kneighbors(rf[rng.choice(len(rf), min(sample, len(rf)), replace=False)])
    real_spacing = float(rr_dist[:, 1].mean())
    coverage = len(np.unique(idx)) / len(rf)
    return {
        'nn_dist_mean': round(nn_dist_mean, 4),
        'nn_dist_ratio': round(nn_dist_mean / (real_spacing + 1e-9), 3),  # ~1 = realistic, <<1 = copies
        'coverage': round(coverage, 4),
    }

rows = []
for r in [100, 500, 1000]:
    tag = f'1_{r}'
    train = pd.read_parquet(os.path.join(PROC_DIR, f'train_{tag}.parquet'))
    real_fraud = train[train['label'] == 1]
    for gen in ['ctgan', 'tvae']:
        aug = pd.read_parquet(os.path.join(AUG_DIR, f'aug_{tag}_{gen}.parquet'))
        # synthetic fraud = the fraud rows added beyond the real ones
        synth_fraud = aug[aug['label'] == 1].iloc[len(real_fraud):]
        m = diversity_metrics(synth_fraud, real_fraud)
        m.update({'ratio': r, 'gen': gen})
        rows.append(m)
        print(f'1:{r} {gen}: nn_ratio={m["nn_dist_ratio"]}, coverage={m["coverage"]}')

div_df = pd.DataFrame(rows)[['ratio','gen','nn_dist_ratio','coverage','nn_dist_mean']]
div_df.to_csv(os.path.join(AUG_DIR, 'synthetic_diversity.csv'), index=False)
div_df

1:100 ctgan: nn_ratio=8.761, coverage=0.156
1:100 tvae: nn_ratio=4.428, coverage=0.1988
1:500 ctgan: nn_ratio=5.76, coverage=0.3016
1:500 tvae: nn_ratio=3.015, coverage=0.326
1:1000 ctgan: nn_ratio=4.802, coverage=0.3394
1:1000 tvae: nn_ratio=2.556, coverage=0.3303


,ratio,gen,nn_dist_ratio,coverage,nn_dist_mean
0,100,ctgan,8.761,0.1560,6.6670
1,100,tvae,4.428,0.1988,3.3695
2,500,ctgan,5.760,0.3016,8.1014
3,500,tvae,3.015,0.3260,4.2400
4,1000,ctgan,4.802,0.3394,8.6973
5,1000,tvae,2.556,0.3303,4.6299
